# Supplementary Figures S4 / S5 -- remaining ten datasets

Bach, Human pancreas, Human PBMC, Muraro, Klein, QS Limb Muscle,
QS Trachea, Romanov, Young and Wang Lung: predicted-cluster UMAPs at each method's inferred k (S4)
and at the annotated reference k (S5).

Split from `Figure3B+S1A+S2A.ipynb`; the three main-text datasets are in
`fig3cd.ipynb`. The setup cells below (imports, `reducer`, `plot_cluster`)
are shared with that notebook.


In [2]:
import pandas as pd
import scanpy as sc
import numpy as np
import h5py
import umap
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use('Agg')
from matplotlib.pyplot import plot,savefig
from sklearn import metrics

import warnings
warnings.filterwarnings("ignore")


import seaborn as sns

/Volumes/SSD/MCW/Research/Aim 1/DMVAE/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='euclidean')

In [5]:
def plot_cluster(df, method_name, y_true, y_true_int, by, ax, y_true2=None, y_true2_int=None):
    """
    df: result object for the method
    method_name: string key in embeddings dict
    y_true: original labels (e.g. strings)
    y_true_int: integer-encoded labels
    by: "pred" or "true"
    ax: matplotlib axis
    """
    if method_name in ('scVI', 'ADClust', 'scAce'):
        y_use_int = y_true_int
    else:
        y_use_int = y_true2_int if y_true2_int is not None else y_true_int

    emb_all = np.asarray(embeddings[method_name])

    if method_name == 'scAce':
        y_pred = df['Clusters'][-1][-1]
    elif method_name == 'ADClust':
        y_pred = df['Clusters']
    else:
        y_pred = df['Clusters']

    y_pred = np.asarray(y_pred, dtype='int').squeeze()
    n = min(len(y_pred), len(y_use_int))
    if len(y_pred) != len(y_use_int):
        y_pred = y_pred[:n]
        emb_all = emb_all[:n]
        y_use_int = y_use_int[:n]
    if isinstance(umap_all, dict) and method_name in umap_all:
        u = umap_all[method_name]
        umap_coords = u[:n] if len(u) > n else u
    else:
        umap_coords = reducer.fit_transform(emb_all)

    if method_name in ('scGMAAE', 'scGNN', 'scDAC', 'DMVAE') or method_name.lower() == 'scvi':
        ari = np.round(df['ARI'], 2) if method_name.lower() != 'scvi' else np.round(df['ARI'], 2)
        nmi = np.round(df['NMI'], 2) if method_name.lower() != 'scvi' else np.round(df['NMI'], 2)
    else:
        ari = np.round(metrics.adjusted_rand_score(y_pred, y_use_int), 2)
        nmi = np.round(metrics.normalized_mutual_info_score(y_pred, y_use_int), 2)
    ari = float(np.atleast_1d(ari).flat[-1])
    nmi = float(np.atleast_1d(nmi).flat[-1])
    print('Method: {}, ARI={}, NMI={}'.format(method_name, ari, nmi))

    adata = sc.AnnData(pd.DataFrame(np.random.rand(len(y_pred), 1)))
    adata.obs['pred'] = y_pred
    adata.obs['pred'] = adata.obs['pred'].astype(str).astype('category')

    adata.obs['true'] = y_use_int
    adata.obs['true'] = adata.obs['true'].astype(str).astype('category')

    '''if method_name == 'scvi':
        adata.obs['true'] = y_true_int_scvi
        adata.obs['true'] = adata.obs['true'].astype(str).astype('category')
    else:
        adata.obs['true'] = y_true_int
        adata.obs['true'] = adata.obs['true'].astype(str).astype('category')'''

    adata.obsm['X_umap'] = umap_coords

    K_pred = len(np.unique(y_pred))
    K_true = len(np.unique(y_use_int))

    if by == "pred":
        sc.pl.umap(adata, color=['pred'], ax=ax, show=False, legend_loc=None, size=8)
        ax.set_title('K = {}   ARI = {:.2f}'.format(K_pred, ari), fontsize=30, family='Arial')
    else:
        sc.pl.umap(adata, color=['true'], ax=ax, show=False, legend_loc=None, size=8)
        ax.set_title('K = {}'.format(K_true), fontsize=30, family='Arial')

    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.tick_params(bottom=False, left=False)

    for spine in ax.spines.values():
        spine.set_visible(False)

    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    ax.plot([xmin, xmax], [ymin, ymin], color="black", linewidth=1)
    ax.plot([xmin, xmin], [ymin, ymax], color="black", linewidth=1)
    ax.set_facecolor("white")

# Supplementary

In [6]:
fig = plt.figure(figsize=(30, 40))
sub_figs = fig.subfigures(10, 1)
axs = []

for i, sub_fig in enumerate(sub_figs):     
    axs.append(sub_fig.subplots(1, 6))
    
axs = np.array(axs)

## Bach

In [7]:
data_mat = h5py.File('/Volumes/SSD/MCW/Research/Aim 1/Data/Bach/data.h5')
obs = data_mat['obs']
ds = obs['cell_type1']
print(ds.dtype, ds.shape)
y_true_bytes = np.array(ds)
y_true = y_true_bytes.astype(str)
classes, y_true_int = np.unique(y_true, return_inverse=True)


|S4 (23184,)


In [8]:

data_mat.close()
scvi = np.load('/Users/enid/Downloads/Bach/scvi.npz', allow_pickle=True)
adclust = np.load('/Users/enid/Downloads/Bach/adclust.npz')
scace = np.load('/Users/enid/Downloads/Bach/scace.npz', allow_pickle=True)
dmvae = np.load('/Users/enid/Downloads/Bach/dmvae.npz')
scdac = np.load('/Users/enid/Downloads/Bach/scdac.npz')
scgnn = np.load('/Users/enid/Downloads/Bach/scgnn.npz')
methods = {
    "scVI": scvi,
    "scGNN": scgnn,
    "ADClust": adclust,
    "scAce": scace,
    "scDAC": scdac,
    "DMVAE": dmvae,
}

embeddings = {}
for name, method in methods.items():
    emb = method["Embedding"]
    # 2D = single matrix (scVI, DMVAE, scGNN, scDAC); 3D = use last (ADClust, scAce)
    embeddings[name] = emb if emb.ndim == 2 else emb[-1]

umap_all = {}
for name, emb in embeddings.items():
    print(name)
    adata = sc.AnnData(emb)
    sc.pp.neighbors(adata)
    sc.tl.umap(adata, random_state=0)
    umap_all[name] = np.array(adata.obsm["X_umap"])
#umap_all = np.load("umap/umap_f3_human.npz")['UMAP']


scVI
scGNN
ADClust
scAce
scDAC
DMVAE


In [9]:
np.savez("/Volumes/SSD/MCW/Research/Aim 1/Results/umap_bach.npz", UMAP=umap_all)

In [ ]:
for j in range(6):
    axs[0][j].clear()
plot_cluster(scvi, 'scVI', y_true, y_true_int, "pred", axs[0][0])
plot_cluster(scgnn, 'scGNN', y_true, y_true_int, "pred", axs[0][1])
plot_cluster(adclust, 'ADClust', y_true, y_true_int, "pred", axs[0][2])
plot_cluster(scace, 'scAce', y_true, y_true_int, "pred", axs[0][3])
plot_cluster(scdac, 'scDAC', y_true, y_true_int, "pred", axs[0][4])
plot_cluster(dmvae, 'DMVAE', y_true, y_true_int, "pred", axs[0][5])


Method: scVI, ARI=0.49, NMI=0.74
Method: scGNN, ARI=0.67, NMI=0.77
Method: ADClust, ARI=0.85, NMI=0.8
Method: scAce, ARI=0.62, NMI=0.75
Method: scDAC, ARI=0.8, NMI=0.8
Method: DMVAE, ARI=0.92, NMI=0.89


## Human pancreas

In [ ]:
# Inspect what data_mat (human_p/data.h5) stores for cell type
data_mat = h5py.File('/Volumes/SSD/MCW/Research/Aim 1/Data/human_p/data.h5')
print("Top-level keys:", list(data_mat.keys()))
if 'obs' in data_mat:
    obs = data_mat['obs']
    print("obs keys:", list(obs.keys()))
    # Cell type is typically in 'cell_type1' (or similar)
    for key in obs.keys():
        if 'cell' in key.lower() or 'type' in key.lower() or 'label' in key.lower():
            d = obs[key]
            arr = np.asarray(d)
            decode = lambda x: x.decode() if isinstance(x, bytes) else x
            print(f"\n  {key}: shape={arr.shape}, dtype={arr.dtype}")
            if arr.size > 0:
                try:
                    uniq = np.unique(arr)
                    if len(uniq) <= 20:
                        print(f"    unique: {[decode(u) for u in uniq]}")
                    else:
                        print(f"    unique (first 20): {[decode(u) for u in uniq[:20]]} ... ({len(uniq)} total)")
                except Exception:
                    print(f"    sample: {arr.flat[0]}")
data_mat.close()

Top-level keys: ['X', 'Y']


In [11]:
# Inspect what data_mat (human_p/data.h5) stores for cell type
data_mat = h5py.File('/Volumes/SSD/MCW/Research/Aim 1/Data/human_p/data.h5')
y_true = y_true_int = np.array(data_mat['Y'], dtype='int')
data_mat.close()

In [12]:
scvi = np.load('/Users/enid/Downloads/human_p/scvi.npz', allow_pickle=True)
adclust = np.load('/Users/enid/Downloads/human_p/adclust.npz')
scace = np.load('/Users/enid/Downloads/human_p/scace.npz', allow_pickle=True)
dmvae = np.load('/Users/enid/Downloads/human_p/dmvae.npz')
scdac = np.load('/Users/enid/Downloads/human_p/scdac.npz')
scgnn = np.load('/Users/enid/Downloads/human_p/scgnn.npz')
methods = {
    "scVI": scvi,
    "scGNN": scgnn,
    "ADClust": adclust,
    "scAce": scace,
    "scDAC": scdac,
    "DMVAE": dmvae,
}

embeddings = {}
for name, method in methods.items():
    emb = method["Embedding"]
    # 2D = single matrix (scVI, DMVAE, scGNN, scDAC); 3D = use last (ADClust, scAce)
    embeddings[name] = emb if emb.ndim == 2 else emb[-1]

umap_all = {}
for name, emb in embeddings.items():
    print(name)
    adata = sc.AnnData(emb)
    sc.pp.neighbors(adata)
    sc.tl.umap(adata, random_state=0)
    umap_all[name] = np.array(adata.obsm["X_umap"])
#umap_all = np.load("umap/umap_f3_human.npz")['UMAP']

scVI
scGNN
ADClust
scAce
scDAC
DMVAE


In [13]:
np.savez("/Volumes/SSD/MCW/Research/Aim 1/Results/umap_human_p.npz", UMAP=umap_all)

In [ ]:
for j in range(6):
    axs[1][j].clear()
plot_cluster(scvi, 'scVI', y_true, y_true_int, "pred", axs[1][0])
plot_cluster(scgnn, 'scGNN', y_true, y_true_int, "pred", axs[1][1])
plot_cluster(adclust, 'ADClust', y_true, y_true_int, "pred", axs[1][2])
plot_cluster(scace, 'scAce', y_true, y_true_int, "pred", axs[1][3])
plot_cluster(scdac, 'scDAC', y_true, y_true_int, "pred", axs[1][4])
plot_cluster(dmvae, 'DMVAE', y_true, y_true_int, "pred", axs[1][5])

Method: scVI, ARI=0.72, NMI=0.84
Method: scGNN, ARI=0.56, NMI=0.59
Method: ADClust, ARI=0.84, NMI=0.8
Method: scAce, ARI=0.89, NMI=0.86
Method: scDAC, ARI=0.84, NMI=0.84
Method: DMVAE, ARI=0.94, NMI=0.88


## Human PBMC

In [15]:
y_true = y_true_int = np.loadtxt("/Volumes/SSD/MCW/Research/Aim 1/Data/PBMC/pbmc_meta_full.txt")

scvi = np.load('/Users/enid/Downloads/PBMC/scvi.npz', allow_pickle=True)
adclust = np.load('/Users/enid/Downloads/PBMC/adclust.npz')
scace = np.load('/Users/enid/Downloads/PBMC/scace.npz', allow_pickle=True)
dmvae = np.load('/Users/enid/Downloads/PBMC/dmvae.npz')
scdac = np.load('/Users/enid/Downloads/PBMC/scdac.npz')
scgnn = np.load('/Users/enid/Downloads/PBMC/scgnn.npz')
methods = {
    "scVI": scvi,
    "scGNN": scgnn,
    "ADClust": adclust,
    "scAce": scace,
    "scDAC": scdac,
    "DMVAE": dmvae,
}

embeddings = {}
for name, method in methods.items():
    emb = method["Embedding"]
    # 2D = single matrix (scVI, DMVAE, scGNN, scDAC); 3D = use last (ADClust, scAce)
    embeddings[name] = emb if emb.ndim == 2 else emb[-1]

umap_all = {}
for name, emb in embeddings.items():
    print(name)
    adata = sc.AnnData(emb)
    sc.pp.neighbors(adata)
    sc.tl.umap(adata, random_state=0)
    umap_all[name] = np.array(adata.obsm["X_umap"])
#umap_all = np.load("umap/umap_f3_human.npz")['UMAP']

scVI
scGNN
ADClust
scAce
scDAC
DMVAE


In [16]:
np.savez("/Volumes/SSD/MCW/Research/Aim 1/Results/umap_pbmc.npz", UMAP=umap_all)

In [17]:
for j in range(6):
    axs[2][j].clear()
plot_cluster(scvi, 'scVI', y_true, y_true_int, "pred", axs[2][0])
plot_cluster(scgnn, 'scGNN', y_true, y_true_int, "pred", axs[2][1])
plot_cluster(adclust, 'ADClust', y_true, y_true_int, "pred", axs[2][2])
plot_cluster(scace, 'scAce', y_true, y_true_int, "pred", axs[2][3])
plot_cluster(scdac, 'scDAC', y_true, y_true_int, "pred", axs[2][4])
plot_cluster(dmvae, 'DMVAE', y_true, y_true_int, "pred", axs[2][5])

Method: scVI, ARI=0.33, NMI=0.58
Method: scGNN, ARI=0.1, NMI=0.32
Method: ADClust, ARI=0.39, NMI=0.58
Method: scAce, ARI=0.49, NMI=0.59
Method: scDAC, ARI=0.17, NMI=0.38
Method: DMVAE, ARI=0.91, NMI=0.87


## Muraro

In [34]:
data_mat = h5py.File('/Volumes/SSD/MCW/Research/Aim 1/Data/Muraro/data.h5')
obs = data_mat['obs']
ds = obs['cell_type1']
print(ds.dtype, ds.shape)
y_true_bytes = np.array(ds)
y_true = y_true_bytes.astype(str)
classes, y_true_int = np.unique(y_true, return_inverse=True)
data_mat.close()

|S12 (2122,)


In [35]:
scvi = np.load('/Users/enid/Downloads/Muraro/scvi.npz', allow_pickle=True)
adclust = np.load('/Users/enid/Downloads/Muraro/adclust.npz')
scace = np.load('/Users/enid/Downloads/Muraro/scace.npz', allow_pickle=True)
dmvae = np.load('/Users/enid/Downloads/Muraro/dmvae.npz')
scdac = np.load('/Users/enid/Downloads/Muraro/scdac.npz')
scgnn = np.load('/Users/enid/Downloads/Muraro/scgnn.npz')
methods = {
    "scVI": scvi,
    "scGNN": scgnn,
    "ADClust": adclust,
    "scAce": scace,
    "scDAC": scdac,
    "DMVAE": dmvae,
}

embeddings = {}
for name, method in methods.items():
    emb = method["Embedding"]
    # 2D = single matrix (scVI, DMVAE, scGNN, scDAC); 3D = use last (ADClust, scAce)
    embeddings[name] = emb if emb.ndim == 2 else emb[-1]

umap_all = {}
for name, emb in embeddings.items():
    print(name)
    adata = sc.AnnData(emb)
    sc.pp.neighbors(adata)
    sc.tl.umap(adata, random_state=0)
    umap_all[name] = np.array(adata.obsm["X_umap"])
#umap_all = np.load("umap/umap_f3_human.npz")['UMAP']


scVI
scGNN
ADClust
scAce
scDAC
DMVAE


In [36]:
np.savez("/Volumes/SSD/MCW/Research/Aim 1/Results/umap_Muraro.npz", UMAP=umap_all)

In [ ]:
for j in range(6):
    axs[3][j].clear()
plot_cluster(scvi, 'scVI', y_true, y_true_int, "pred", axs[4][0])
plot_cluster(scgnn, 'scGNN', y_true, y_true_int, "pred", axs[4][1])
plot_cluster(adclust, 'ADClust', y_true, y_true_int, "pred", axs[4][2])
plot_cluster(scace, 'scAce', y_true, y_true_int, "pred", axs[4][3])
plot_cluster(scdac, 'scDAC', y_true, y_true_int, "pred", axs[4][4])
plot_cluster(dmvae, 'DMVAE', y_true, y_true_int, "pred", axs[4][5])

Method: scVI, ARI=0.47, NMI=0.74
Method: scGNN, ARI=0.5, NMI=0.62
Method: ADClust, ARI=0.82, NMI=0.81
Method: scAce, ARI=0.93, NMI=0.88
Method: scDAC, ARI=0.65, NMI=0.79
Method: DMVAE, ARI=0.9, NMI=0.84


## Klein

In [ ]:
data_mat = h5py.File('/Volumes/SSD/MCW/Research/Aim 1/Data/mouse_e/data.h5')
obs = data_mat['obs']
ds = obs['cell_type1']
print(ds.dtype, ds.shape)
y_true_bytes = np.array(ds)
y_true = y_true_bytes.astype(str)
classes, y_true_int = np.unique(y_true, return_inverse=True)
data_mat.close()

|S3 (2717,)


In [23]:
scvi = np.load('/Users/enid/Downloads/Klein/scvi.npz', allow_pickle=True)
adclust = np.load('/Users/enid/Downloads/Klein/adclust.npz')
scace = np.load('/Users/enid/Downloads/Klein/scace.npz', allow_pickle=True)
dmvae = np.load('/Users/enid/Downloads/Klein/dmvae.npz')
scdac = np.load('/Users/enid/Downloads/Klein/scdac.npz')
scgnn = np.load('/Users/enid/Downloads/Klein/scgnn.npz')
methods = {
    "scVI": scvi,
    "scGNN": scgnn,
    "ADClust": adclust,
    "scAce": scace,
    "scDAC": scdac,
    "DMVAE": dmvae,
}

embeddings = {}
for name, method in methods.items():
    emb = method["Embedding"]
    # 2D = single matrix (scVI, DMVAE, scGNN, scDAC); 3D = use last (ADClust, scAce)
    embeddings[name] = emb if emb.ndim == 2 else emb[-1]

umap_all = {}
for name, emb in embeddings.items():
    print(name)
    adata = sc.AnnData(emb)
    sc.pp.neighbors(adata)
    sc.tl.umap(adata, random_state=0)
    umap_all[name] = np.array(adata.obsm["X_umap"])
#umap_all = np.load("umap/umap_f3_human.npz")['UMAP']


scVI
scGNN
ADClust
scAce
scDAC
DMVAE


In [24]:
np.savez("/Volumes/SSD/MCW/Research/Aim 1/Results/umap_Klein.npz", UMAP=umap_all)

In [ ]:
for j in range(6):
    axs[4][j].clear()
plot_cluster(scvi, 'scVI', y_true, y_true_int, "pred", axs[3][0])
plot_cluster(scgnn, 'scGNN', y_true, y_true_int, "pred", axs[3][1])
plot_cluster(adclust, 'ADClust', y_true, y_true_int, "pred", axs[3][2])
plot_cluster(scace, 'scAce', y_true, y_true_int, "pred", axs[3][3])
plot_cluster(scdac, 'scDAC', y_true, y_true_int, "pred", axs[3][4])
plot_cluster(dmvae, 'DMVAE', y_true, y_true_int, "pred", axs[3][5])

Method: scVI, ARI=0.65, NMI=0.76
Method: scGNN, ARI=0.66, NMI=0.72
Method: ADClust, ARI=0.72, NMI=0.68
Method: scAce, ARI=0.9, NMI=0.91
Method: scDAC, ARI=0.55, NMI=0.7
Method: DMVAE, ARI=0.84, NMI=0.82


## QS_LM

In [26]:
data_mat = h5py.File('/Volumes/SSD/MCW/Research/Aim 1/Data/Quake_Smart-seq2_Limb_Muscle/data.h5')
obs = data_mat['obs']
ds = obs['cell_type1']
print(ds.dtype, ds.shape)
y_true_bytes = np.array(ds)
y_true = y_true_bytes.astype(str)
classes, y_true_int = np.unique(y_true, return_inverse=True)
data_mat.close()

|S31 (1090,)


In [27]:
scvi = np.load('/Users/enid/Downloads/QS_LM/scvi.npz', allow_pickle=True)
adclust = np.load('/Users/enid/Downloads/QS_LM/adclust.npz')
scace = np.load('/Users/enid/Downloads/QS_LM/scace.npz', allow_pickle=True)
dmvae = np.load('/Users/enid/Downloads/QS_LM/dmvae.npz')
scdac = np.load('/Users/enid/Downloads/QS_LM/scdac.npz')
scgnn = np.load('/Users/enid/Downloads/QS_LM/scgnn.npz')
methods = {
    "scVI": scvi,
    "scGNN": scgnn,
    "ADClust": adclust,
    "scAce": scace,
    "scDAC": scdac,
    "DMVAE": dmvae,
}

embeddings = {}
for name, method in methods.items():
    emb = method["Embedding"]
    # 2D = single matrix (scVI, DMVAE, scGNN, scDAC); 3D = use last (ADClust, scAce)
    embeddings[name] = emb if emb.ndim == 2 else emb[-1]

umap_all = {}
for name, emb in embeddings.items():
    print(name)
    adata = sc.AnnData(emb)
    sc.pp.neighbors(adata)
    sc.tl.umap(adata, random_state=0)
    umap_all[name] = np.array(adata.obsm["X_umap"])
#umap_all = np.load("umap/umap_f3_human.npz")['UMAP']


scVI
scGNN
ADClust
scAce
scDAC
DMVAE


In [28]:
np.savez("/Volumes/SSD/MCW/Research/Aim 1/Results/umap_QS_LM.npz", UMAP=umap_all)

In [ ]:
for j in range(6):
    axs[5][j].clear()
plot_cluster(scvi, 'scVI', y_true, y_true_int, "pred", axs[5][0])
plot_cluster(scgnn, 'scGNN', y_true, y_true_int, "pred", axs[5][1])
plot_cluster(adclust, 'ADClust', y_true, y_true_int, "pred", axs[5][2])
plot_cluster(scace, 'scAce', y_true, y_true_int, "pred", axs[5][3])
plot_cluster(scdac, 'scDAC', y_true, y_true_int, "pred", axs[5][4])
plot_cluster(dmvae, 'DMVAE', y_true, y_true_int, "pred", axs[5][5])

Method: scVI, ARI=0.48, NMI=0.75
Method: scGNN, ARI=0.54, NMI=0.72
Method: ADClust, ARI=0.97, NMI=0.95
Method: scAce, ARI=0.64, NMI=0.78
Method: scDAC, ARI=0.55, NMI=0.74
Method: DMVAE, ARI=0.92, NMI=0.85


## QS_trachea

In [38]:
data_mat = h5py.File('/Volumes/SSD/MCW/Research/Aim 1/Data/Quake_Smart-seq2_Trachea/data.h5')
obs = data_mat['obs']
ds = obs['cell_type1']
print(ds.dtype, ds.shape)
y_true_bytes = np.array(ds)
y_true = y_true_bytes.astype(str)
classes, y_true_int = np.unique(y_true, return_inverse=True)
data_mat.close()

|S17 (1350,)


In [39]:
scvi = np.load('/Users/enid/Downloads/QS_trachea/scvi.npz', allow_pickle=True)
adclust = np.load('/Users/enid/Downloads/QS_trachea/adclust.npz')
scace = np.load('/Users/enid/Downloads/QS_trachea/scace.npz', allow_pickle=True)
dmvae = np.load('/Users/enid/Downloads/QS_trachea/dmvae.npz')
scdac = np.load('/Users/enid/Downloads/QS_trachea/scdac.npz')
scgnn = np.load('/Users/enid/Downloads/QS_trachea/scgnn.npz')
methods = {
    "scVI": scvi,
    "scGNN": scgnn,
    "ADClust": adclust,
    "scAce": scace,
    "scDAC": scdac,
    "DMVAE": dmvae,
}

embeddings = {}
for name, method in methods.items():
    emb = method["Embedding"]
    # 2D = single matrix (scVI, DMVAE, scGNN, scDAC); 3D = use last (ADClust, scAce)
    embeddings[name] = emb if emb.ndim == 2 else emb[-1]

umap_all = {}
for name, emb in embeddings.items():
    print(name)
    adata = sc.AnnData(emb)
    sc.pp.neighbors(adata)
    sc.tl.umap(adata, random_state=0)
    umap_all[name] = np.array(adata.obsm["X_umap"])
#umap_all = np.load("umap/umap_f3_human.npz")['UMAP']


scVI
scGNN
ADClust
scAce
scDAC
DMVAE


In [40]:
np.savez("/Volumes/SSD/MCW/Research/Aim 1/Results/umap_QS_trachea.npz", UMAP=umap_all)

In [41]:
for j in range(6):
    axs[6][j].clear()
plot_cluster(scvi, 'scVI', y_true, y_true_int, "pred", axs[6][0])
plot_cluster(scgnn, 'scGNN', y_true, y_true_int, "pred", axs[6][1])
plot_cluster(adclust, 'ADClust', y_true, y_true_int, "pred", axs[6][2])
plot_cluster(scace, 'scAce', y_true, y_true_int, "pred", axs[6][3])
plot_cluster(scdac, 'scDAC', y_true, y_true_int, "pred", axs[6][4])
plot_cluster(dmvae, 'DMVAE', y_true, y_true_int, "pred", axs[6][5])

Method: scVI, ARI=0.16, NMI=0.5
Method: scGNN, ARI=0.23, NMI=0.57
Method: ADClust, ARI=0.52, NMI=0.59
Method: scAce, ARI=0.34, NMI=0.61
Method: scDAC, ARI=0.37, NMI=0.65
Method: DMVAE, ARI=0.88, NMI=0.79


## Romanov

In [42]:
data_mat = h5py.File('/Volumes/SSD/MCW/Research/Aim 1/Data/Romanov/data.h5')
obs = data_mat['obs']
ds = obs['cell_type1']
print(ds.dtype, ds.shape)
y_true_bytes = np.array(ds)
y_true = y_true_bytes.astype(str)
classes, y_true_int = np.unique(y_true, return_inverse=True)
data_mat.close()

|S12 (2881,)


In [43]:
scvi = np.load('/Users/enid/Downloads/Romanov/scvi.npz', allow_pickle=True)
adclust = np.load('/Users/enid/Downloads/Romanov/adclust.npz')
scace = np.load('/Users/enid/Downloads/Romanov/scace.npz', allow_pickle=True)
dmvae = np.load('/Users/enid/Downloads/Romanov/dmvae.npz')
scdac = np.load('/Users/enid/Downloads/Romanov/scdac.npz')
scgnn = np.load('/Users/enid/Downloads/Romanov/scgnn.npz')
methods = {
    "scVI": scvi,
    "scGNN": scgnn,
    "ADClust": adclust,
    "scAce": scace,
    "scDAC": scdac,
    "DMVAE": dmvae,
}

embeddings = {}
for name, method in methods.items():
    emb = method["Embedding"]
    # 2D = single matrix (scVI, DMVAE, scGNN, scDAC); 3D = use last (ADClust, scAce)
    embeddings[name] = emb if emb.ndim == 2 else emb[-1]

umap_all = {}
for name, emb in embeddings.items():
    print(name)
    adata = sc.AnnData(emb)
    sc.pp.neighbors(adata)
    sc.tl.umap(adata, random_state=0)
    umap_all[name] = np.array(adata.obsm["X_umap"])
#umap_all = np.load("umap/umap_f3_human.npz")['UMAP']


scVI
scGNN
ADClust
scAce
scDAC
DMVAE


In [44]:
np.savez("/Volumes/SSD/MCW/Research/Aim 1/Results/umap_Romanov.npz", UMAP=umap_all)

In [45]:
for j in range(6):
    axs[7][j].clear()
plot_cluster(scvi, 'scVI', y_true, y_true_int, "pred", axs[7][0])
plot_cluster(scgnn, 'scGNN', y_true, y_true_int, "pred", axs[7][1])
plot_cluster(adclust, 'ADClust', y_true, y_true_int, "pred", axs[7][2])
plot_cluster(scace, 'scAce', y_true, y_true_int, "pred", axs[7][3])
plot_cluster(scdac, 'scDAC', y_true, y_true_int, "pred", axs[7][4])
plot_cluster(dmvae, 'DMVAE', y_true, y_true_int, "pred", axs[7][5])

Method: scVI, ARI=0.3, NMI=0.56
Method: scGNN, ARI=0.26, NMI=0.33
Method: ADClust, ARI=0.33, NMI=0.45
Method: scAce, ARI=0.41, NMI=0.57
Method: scDAC, ARI=0.36, NMI=0.53
Method: DMVAE, ARI=0.71, NMI=0.6


## Young

In [46]:
data_mat = h5py.File('/Volumes/SSD/MCW/Research/Aim 1/Data/Young/data.h5')
obs = data_mat['obs']
ds = obs['cell_type1']
print(ds.dtype, ds.shape)
y_true_bytes = np.array(ds)
y_true = y_true_bytes.astype(str)
classes, y_true_int = np.unique(y_true, return_inverse=True)
data_mat.close()

|S24 (5685,)


In [47]:
scvi = np.load('/Users/enid/Downloads/Young/scvi.npz', allow_pickle=True)
adclust = np.load('/Users/enid/Downloads/Young/adclust.npz')
scace = np.load('/Users/enid/Downloads/Young/scace.npz', allow_pickle=True)
dmvae = np.load('/Users/enid/Downloads/Young/dmvae.npz')
scdac = np.load('/Users/enid/Downloads/Young/scdac.npz')
scgnn = np.load('/Users/enid/Downloads/Young/scgnn.npz')
methods = {
    "scVI": scvi,
    "scGNN": scgnn,
    "ADClust": adclust,
    "scAce": scace,
    "scDAC": scdac,
    "DMVAE": dmvae,
}

embeddings = {}
for name, method in methods.items():
    emb = method["Embedding"]
    # 2D = single matrix (scVI, DMVAE, scGNN, scDAC); 3D = use last (ADClust, scAce)
    embeddings[name] = emb if emb.ndim == 2 else emb[-1]

umap_all = {}
for name, emb in embeddings.items():
    print(name)
    adata = sc.AnnData(emb)
    sc.pp.neighbors(adata)
    sc.tl.umap(adata, random_state=0)
    umap_all[name] = np.array(adata.obsm["X_umap"])
#umap_all = np.load("umap/umap_f3_human.npz")['UMAP']


scVI
scGNN
ADClust
scAce
scDAC
DMVAE


In [48]:
np.savez("/Volumes/SSD/MCW/Research/Aim 1/Results/umap_Young.npz", UMAP=umap_all)

In [ ]:
for j in range(6):
    axs[8][j].clear()
plot_cluster(scvi, 'scVI', y_true, y_true_int, "pred", axs[9][0])
plot_cluster(scgnn, 'scGNN', y_true, y_true_int, "pred", axs[9][1])
plot_cluster(adclust, 'ADClust', y_true, y_true_int, "pred", axs[9][2])
plot_cluster(scace, 'scAce', y_true, y_true_int, "pred", axs[9][3])
plot_cluster(scdac, 'scDAC', y_true, y_true_int, "pred", axs[9][4])
plot_cluster(dmvae, 'DMVAE', y_true, y_true_int, "pred", axs[9][5])

Method: scVI, ARI=0.47, NMI=0.72
Method: scGNN, ARI=0.23, NMI=0.37
Method: ADClust, ARI=0.39, NMI=0.57
Method: scAce, ARI=0.6, NMI=0.74
Method: scDAC, ARI=0.45, NMI=0.63
Method: DMVAE, ARI=0.56, NMI=0.66


## Wang Lung

In [50]:
data_mat = h5py.File('/Volumes/SSD/MCW/Research/Aim 1/Data/Wang_Lung/data.h5')
obs = data_mat['obs']
ds = obs['cell_type1']
print(ds.dtype, ds.shape)
y_true_bytes = np.array(ds)
y_true = y_true_bytes.astype(str)
classes, y_true_int = np.unique(y_true, return_inverse=True)
data_mat.close()

|S16 (9519,)


In [52]:
scvi = np.load('/Users/enid/Downloads/Wang_Lung/scvi.npz', allow_pickle=True)
adclust = np.load('/Users/enid/Downloads/Wang_Lung/adclust.npz')
scace = np.load('/Users/enid/Downloads/Wang_Lung/scace.npz', allow_pickle=True)
dmvae = np.load('/Users/enid/Downloads/Wang_Lung/dmvae.npz')
scdac = np.load('/Users/enid/Downloads/Wang_Lung/scdac.npz')
scgnn = np.load('/Users/enid/Downloads/Wang_Lung/scgnn.npz')
methods = {
    "scVI": scvi,
    "scGNN": scgnn,
    "ADClust": adclust,
    "scAce": scace,
    "scDAC": scdac,
    "DMVAE": dmvae,
}

embeddings = {}
for name, method in methods.items():
    emb = method["Embedding"]
    # 2D = single matrix (scVI, DMVAE, scGNN, scDAC); 3D = use last (ADClust, scAce)
    embeddings[name] = emb if emb.ndim == 2 else emb[-1]

umap_all = {}
for name, emb in embeddings.items():
    print(name)
    adata = sc.AnnData(emb)
    sc.pp.neighbors(adata)
    sc.tl.umap(adata, random_state=0)
    umap_all[name] = np.array(adata.obsm["X_umap"])
#umap_all = np.load("umap/umap_f3_human.npz")['UMAP']


scVI
scGNN
ADClust
scAce
scDAC
DMVAE


In [53]:
np.savez("/Volumes/SSD/MCW/Research/Aim 1/Results/umap_Wang_Lung.npz", UMAP=umap_all)

In [ ]:
for j in range(6):
    axs[9][j].clear()
plot_cluster(scvi, 'scVI', y_true, y_true_int, "pred", axs[8][0])
plot_cluster(scgnn, 'scGNN', y_true, y_true_int, "pred", axs[8][1])
plot_cluster(adclust, 'ADClust', y_true, y_true_int, "pred", axs[8][2])
plot_cluster(scace, 'scAce', y_true, y_true_int, "pred", axs[8][3])
plot_cluster(scdac, 'scDAC', y_true, y_true_int, "pred", axs[8][4])
plot_cluster(dmvae, 'DMVAE', y_true, y_true_int, "pred", axs[8][5])

Method: scVI, ARI=0.14, NMI=0.37
Method: scGNN, ARI=0.22, NMI=0.42
Method: ADClust, ARI=0.37, NMI=0.52
Method: scAce, ARI=0.19, NMI=0.37
Method: scDAC, ARI=0.23, NMI=0.4
Method: DMVAE, ARI=0.96, NMI=0.9


## save the plot

In [55]:
plt.savefig('/Volumes/SSD/MCW/Research/Aim 1/Documents/Paper_draft/papers/umap_pred_suppl.png', dpi=600, format='png', bbox_inches='tight')

# True label

In [56]:
fig = plt.figure(figsize=(30, 40))
sub_figs = fig.subfigures(10, 1)
axs = []

for i, sub_fig in enumerate(sub_figs):     
    axs.append(sub_fig.subplots(1, 6))
    
axs = np.array(axs)

## Bach

In [57]:
umap_all = np.load("/Volumes/SSD/MCW/Research/Aim 1/Results/umap_bach.npz", allow_pickle=True)['UMAP']
umap_all = umap_all.item()
data_mat = h5py.File('/Volumes/SSD/MCW/Research/Aim 1/Data/Bach/data.h5')
obs = data_mat['obs']
ds = obs['cell_type1']
print(ds.dtype, ds.shape)
y_true_bytes = np.array(ds)
y_true = y_true_bytes.astype(str)
classes, y_true_int = np.unique(y_true, return_inverse=True)

data_mat.close()

scvi = np.load('/Users/enid/Downloads/Bach/scvi.npz', allow_pickle=True)
adclust = np.load('/Users/enid/Downloads/Bach/adclust.npz')
scace = np.load('/Users/enid/Downloads/Bach/scace.npz', allow_pickle=True)
dmvae = np.load('/Users/enid/Downloads/Bach/dmvae.npz')
scgnn = np.load('/Users/enid/Downloads/Bach/scgnn.npz')
scdac = np.load('/Users/enid/Downloads/Bach/scdac.npz')

plot_cluster(scvi, 'scVI', y_true, y_true_int, "true", axs[0][0])
plot_cluster(scgnn, 'scGNN', y_true, y_true_int, "true", axs[0][1])
plot_cluster(adclust, 'ADClust', y_true, y_true_int, "true", axs[0][2])
plot_cluster(scace, 'scAce', y_true, y_true_int, "true", axs[0][3])
plot_cluster(scdac, 'scDAC', y_true, y_true_int, "true", axs[0][4])
plot_cluster(dmvae, 'DMVAE', y_true, y_true_int, "true", axs[0][5])

|S4 (23184,)
Method: scVI, ARI=0.49, NMI=0.74
Method: scGNN, ARI=0.67, NMI=0.77
Method: ADClust, ARI=0.85, NMI=0.8
Method: scAce, ARI=0.62, NMI=0.75
Method: scDAC, ARI=0.8, NMI=0.8
Method: DMVAE, ARI=0.92, NMI=0.89


## Human pancreas

In [59]:
umap_all = np.load("/Volumes/SSD/MCW/Research/Aim 1/Results/umap_human_p.npz", allow_pickle=True)['UMAP']
umap_all = umap_all.item()
data_mat = h5py.File('/Volumes/SSD/MCW/Research/Aim 1/Data/human_p/data.h5')
y_true = y_true_int = np.array(data_mat['Y'], dtype='int')
data_mat.close()

scvi = np.load('/Users/enid/Downloads/human_p/scvi.npz', allow_pickle=True)
adclust = np.load('/Users/enid/Downloads/human_p/adclust.npz')
scace = np.load('/Users/enid/Downloads/human_p/scace.npz', allow_pickle=True)
dmvae = np.load('/Users/enid/Downloads/human_p/dmvae.npz')
scgnn = np.load('/Users/enid/Downloads/human_p/scgnn.npz')
scdac = np.load('/Users/enid/Downloads/human_p/scdac.npz')

plot_cluster(scvi, 'scVI', y_true, y_true_int, "true", axs[1][0])
plot_cluster(scgnn, 'scGNN', y_true, y_true_int, "true", axs[1][1])
plot_cluster(adclust, 'ADClust', y_true, y_true_int, "true", axs[1][2])
plot_cluster(scace, 'scAce', y_true, y_true_int, "true", axs[1][3])
plot_cluster(scdac, 'scDAC', y_true, y_true_int, "true", axs[1][4])
plot_cluster(dmvae, 'DMVAE', y_true, y_true_int, "true", axs[1][5])

Method: scVI, ARI=0.72, NMI=0.84
Method: scGNN, ARI=0.56, NMI=0.59
Method: ADClust, ARI=0.84, NMI=0.8
Method: scAce, ARI=0.89, NMI=0.86
Method: scDAC, ARI=0.84, NMI=0.84
Method: DMVAE, ARI=0.94, NMI=0.88


## Human PBMC

In [82]:
def plot_cluster(df, method_name, y_true, y_true_int, by, ax, y_true2=None, y_true2_int=None):
    """
    df: result object for the method
    method_name: string key in embeddings dict
    y_true: original labels (e.g. strings)
    y_true_int: integer-encoded labels
    by: "pred" or "true"
    ax: matplotlib axis
    """
    if method_name in ('scVI', 'ADClust', 'scAce'):
        y_use_int = y_true_int
    elif method_name == "DMVAE":
        y_use_int = y_true_int2 if y_true_int2 is not None else y_true_int
    else:
        y_use_int = y_true_int3 if y_true_int3 is not None else y_true_int

    emb_all = np.asarray(embeddings[method_name])

    if method_name == 'scAce':
        y_pred = df['Clusters'][-1][-1]
    elif method_name == 'ADClust':
        y_pred = df['Clusters']
    else:
        y_pred = df['Clusters']

    y_pred = np.asarray(y_pred, dtype='int').squeeze()
    n = min(len(y_pred), len(y_use_int))
    if len(y_pred) != len(y_use_int):
        y_pred = y_pred[:n]
        emb_all = emb_all[:n]
        y_use_int = y_use_int[:n]
    if isinstance(umap_all, dict) and method_name in umap_all:
        u = umap_all[method_name]
        umap_coords = u[:n] if len(u) > n else u
    else:
        umap_coords = reducer.fit_transform(emb_all)

    if method_name in ('scGMAAE', 'scGNN', 'scDAC', 'DMVAE') or method_name.lower() == 'scvi':
        ari = np.round(df['ARI'], 2) if method_name.lower() != 'scvi' else np.round(df['ARI'], 2)
        nmi = np.round(df['NMI'], 2) if method_name.lower() != 'scvi' else np.round(df['NMI'], 2)
    else:
        ari = np.round(metrics.adjusted_rand_score(y_pred, y_use_int), 2)
        nmi = np.round(metrics.normalized_mutual_info_score(y_pred, y_use_int), 2)
    ari = float(np.atleast_1d(ari).flat[-1])
    nmi = float(np.atleast_1d(nmi).flat[-1])
    print('Method: {}, ARI={}, NMI={}'.format(method_name, ari, nmi))

    adata = sc.AnnData(pd.DataFrame(np.random.rand(len(y_pred), 1)))
    adata.obs['pred'] = y_pred
    adata.obs['pred'] = adata.obs['pred'].astype(str).astype('category')

    adata.obs['true'] = y_use_int
    adata.obs['true'] = adata.obs['true'].astype(str).astype('category')

    '''if method_name == 'scvi':
        adata.obs['true'] = y_true_int_scvi
        adata.obs['true'] = adata.obs['true'].astype(str).astype('category')
    else:
        adata.obs['true'] = y_true_int
        adata.obs['true'] = adata.obs['true'].astype(str).astype('category')'''

    adata.obsm['X_umap'] = umap_coords

    K_pred = len(np.unique(y_pred))
    K_true = len(np.unique(y_use_int))

    if by == "pred":
        sc.pl.umap(adata, color=['pred'], ax=ax, show=False, legend_loc=None, size=8)
        ax.set_title('K = {}   ARI = {:.2f}'.format(K_pred, ari), fontsize=30, family='Arial')
    else:
        sc.pl.umap(adata, color=['true'], ax=ax, show=False, legend_loc=None, size=8)
        ax.set_title('K = {}'.format(K_true), fontsize=30, family='Arial')

    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.tick_params(bottom=False, left=False)

    for spine in ax.spines.values():
        spine.set_visible(False)

    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    ax.plot([xmin, xmax], [ymin, ymin], color="black", linewidth=1)
    ax.plot([xmin, xmin], [ymin, ymax], color="black", linewidth=1)
    ax.set_facecolor("white")

In [83]:
umap_all = np.load("/Volumes/SSD/MCW/Research/Aim 1/Results/umap_pbmc.npz", allow_pickle=True)['UMAP']
umap_all = umap_all.item()
y_true = y_true_int = np.loadtxt("/Volumes/SSD/MCW/Research/Aim 1/Data/PBMC/pbmc_meta_full.txt")
y_true2=y_true_int2=np.loadtxt("/Volumes/SSD/MCW/Research/Aim 1/Data/PBMC/data_celltype.txt")
y_true3=y_true_int3=np.loadtxt("/Volumes/SSD/MCW/Research/Aim 1/Data/PBMC/celltype.txt", skiprows=1, usecols=1, dtype=int)

scvi = np.load('/Users/enid/Downloads/PBMC/scvi.npz', allow_pickle=True)
adclust = np.load('/Users/enid/Downloads/PBMC/adclust.npz')
scace = np.load('/Users/enid/Downloads/PBMC/scace.npz', allow_pickle=True)
dmvae = np.load('/Users/enid/Downloads/PBMC/dmvae.npz')
scgnn = np.load('/Users/enid/Downloads/PBMC/scgnn.npz')
scdac = np.load('/Users/enid/Downloads/PBMC/scdac.npz')

for j in range(6):
    axs[2][j].clear()
plot_cluster(scvi, 'scVI', y_true, y_true_int, "true", axs[2][0])
plot_cluster(scgnn, 'scGNN', y_true, y_true_int, "true", axs[2][1])
plot_cluster(adclust, 'ADClust', y_true, y_true_int, "true", axs[2][2])
plot_cluster(scace, 'scAce', y_true, y_true_int, "true", axs[2][3])
plot_cluster(scdac, 'scDAC', y_true, y_true_int, "true", axs[2][4])
plot_cluster(dmvae, 'DMVAE', y_true, y_true_int, "true", axs[2][5])

Method: scVI, ARI=0.33, NMI=0.58
Method: scGNN, ARI=0.1, NMI=0.32
Method: ADClust, ARI=0.39, NMI=0.58
Method: scAce, ARI=0.49, NMI=0.59
Method: scDAC, ARI=0.17, NMI=0.38
Method: DMVAE, ARI=0.91, NMI=0.87


## Muraro

In [ ]:
umap_all = np.load("/Volumes/SSD/MCW/Research/Aim 1/Results/umap_Muraro.npz", allow_pickle=True)['UMAP']
umap_all = umap_all.item()
data_mat = h5py.File('/Volumes/SSD/MCW/Research/Aim 1/Data/Muraro/data.h5')
obs = data_mat['obs']
ds = obs['cell_type1']
print(ds.dtype, ds.shape)
y_true_bytes = np.array(ds)
y_true = y_true_bytes.astype(str)
classes, y_true_int = np.unique(y_true, return_inverse=True)

data_mat.close()

scvi = np.load('/Users/enid/Downloads/Muraro/scvi.npz', allow_pickle=True)
adclust = np.load('/Users/enid/Downloads/Muraro/adclust.npz')
scace = np.load('/Users/enid/Downloads/Muraro/scace.npz', allow_pickle=True)
dmvae = np.load('/Users/enid/Downloads/Muraro/dmvae.npz')
scgnn = np.load('/Users/enid/Downloads/Muraro/scgnn.npz')
scdac = np.load('/Users/enid/Downloads/Muraro/scdac.npz')

plot_cluster(scvi, 'scVI', y_true, y_true_int, "true", axs[4][0])
plot_cluster(scgnn, 'scGNN', y_true, y_true_int, "true", axs[4][1])
plot_cluster(adclust, 'ADClust', y_true, y_true_int, "true", axs[4][2])
plot_cluster(scace, 'scAce', y_true, y_true_int, "true", axs[4][3])
plot_cluster(scdac, 'scDAC', y_true, y_true_int, "true", axs[4][4])
plot_cluster(dmvae, 'DMVAE', y_true, y_true_int, "true", axs[4][5])

|S12 (2122,)
Method: scVI, ARI=0.47, NMI=0.74
Method: scGNN, ARI=0.5, NMI=0.62
Method: ADClust, ARI=0.82, NMI=0.81
Method: scAce, ARI=0.93, NMI=0.88
Method: scDAC, ARI=0.65, NMI=0.79
Method: DMVAE, ARI=0.9, NMI=0.84


## Klein

In [ ]:
umap_all = np.load("/Volumes/SSD/MCW/Research/Aim 1/Results/umap_Klein.npz", allow_pickle=True)['UMAP']
umap_all = umap_all.item()
data_mat = h5py.File('/Volumes/SSD/MCW/Research/Aim 1/Data/mouse_e/data.h5')
obs = data_mat['obs']
ds = obs['cell_type1']
print(ds.dtype, ds.shape)
y_true_bytes = np.array(ds)
y_true = y_true_bytes.astype(str)
classes, y_true_int = np.unique(y_true, return_inverse=True)

data_mat.close()

scvi = np.load('/Users/enid/Downloads/Klein/scvi.npz', allow_pickle=True)
adclust = np.load('/Users/enid/Downloads/Klein/adclust.npz')
scace = np.load('/Users/enid/Downloads/Klein/scace.npz', allow_pickle=True)
dmvae = np.load('/Users/enid/Downloads/Klein/dmvae.npz')
scgnn = np.load('/Users/enid/Downloads/Klein/scgnn.npz')
scdac = np.load('/Users/enid/Downloads/Klein/scdac.npz')

for j in range(6):
    axs[4][j].clear()
plot_cluster(scvi, 'scVI', y_true, y_true_int, "true", axs[3][0])
plot_cluster(scgnn, 'scGNN', y_true, y_true_int, "true", axs[3][1])
plot_cluster(adclust, 'ADClust', y_true, y_true_int, "true", axs[3][2])
plot_cluster(scace, 'scAce', y_true, y_true_int, "true", axs[3][3])
plot_cluster(scdac, 'scDAC', y_true, y_true_int, "true", axs[3][4])
plot_cluster(dmvae, 'DMVAE', y_true, y_true_int, "true", axs[3][5])

|S3 (2717,)
Method: scVI, ARI=0.65, NMI=0.76
Method: scGNN, ARI=0.66, NMI=0.72
Method: ADClust, ARI=0.72, NMI=0.68
Method: scAce, ARI=0.9, NMI=0.91
Method: scDAC, ARI=0.55, NMI=0.7
Method: DMVAE, ARI=0.84, NMI=0.82


## QS_LM

In [65]:
umap_all = np.load("/Volumes/SSD/MCW/Research/Aim 1/Results/umap_QS_LM.npz", allow_pickle=True)['UMAP']
umap_all = umap_all.item()
data_mat = h5py.File('/Volumes/SSD/MCW/Research/Aim 1/Data/Quake_Smart-seq2_Limb_Muscle/data.h5')
obs = data_mat['obs']
ds = obs['cell_type1']
print(ds.dtype, ds.shape)
y_true_bytes = np.array(ds)
y_true = y_true_bytes.astype(str)
classes, y_true_int = np.unique(y_true, return_inverse=True)

data_mat.close()

scvi = np.load('/Users/enid/Downloads/QS_LM/scvi.npz', allow_pickle=True)
adclust = np.load('/Users/enid/Downloads/QS_LM/adclust.npz')
scace = np.load('/Users/enid/Downloads/QS_LM/scace.npz', allow_pickle=True)
dmvae = np.load('/Users/enid/Downloads/QS_LM/dmvae.npz')
scgnn = np.load('/Users/enid/Downloads/QS_LM/scgnn.npz')
scdac = np.load('/Users/enid/Downloads/QS_LM/scdac.npz')

plot_cluster(scvi, 'scVI', y_true, y_true_int, "true", axs[5][0])
plot_cluster(scgnn, 'scGNN', y_true, y_true_int, "true", axs[5][1])
plot_cluster(adclust, 'ADClust', y_true, y_true_int, "true", axs[5][2])
plot_cluster(scace, 'scAce', y_true, y_true_int, "true", axs[5][3])
plot_cluster(scdac, 'scDAC', y_true, y_true_int, "true", axs[5][4])
plot_cluster(dmvae, 'DMVAE', y_true, y_true_int, "true", axs[5][5])

|S31 (1090,)
Method: scVI, ARI=0.48, NMI=0.75
Method: scGNN, ARI=0.54, NMI=0.72
Method: ADClust, ARI=0.97, NMI=0.95
Method: scAce, ARI=0.64, NMI=0.78
Method: scDAC, ARI=0.55, NMI=0.74
Method: DMVAE, ARI=0.92, NMI=0.85


## QS_trachea

In [66]:
umap_all = np.load("/Volumes/SSD/MCW/Research/Aim 1/Results/umap_QS_trachea.npz", allow_pickle=True)['UMAP']
umap_all = umap_all.item()
data_mat = h5py.File('/Volumes/SSD/MCW/Research/Aim 1/Data/Quake_Smart-seq2_Trachea/data.h5')
obs = data_mat['obs']
ds = obs['cell_type1']
print(ds.dtype, ds.shape)
y_true_bytes = np.array(ds)
y_true = y_true_bytes.astype(str)
classes, y_true_int = np.unique(y_true, return_inverse=True)

data_mat.close()

scvi = np.load('/Users/enid/Downloads/QS_trachea/scvi.npz', allow_pickle=True)
adclust = np.load('/Users/enid/Downloads/QS_trachea/adclust.npz')
scace = np.load('/Users/enid/Downloads/QS_trachea/scace.npz', allow_pickle=True)
dmvae = np.load('/Users/enid/Downloads/QS_trachea/dmvae.npz')
scgnn = np.load('/Users/enid/Downloads/QS_trachea/scgnn.npz')
scdac = np.load('/Users/enid/Downloads/QS_trachea/scdac.npz')

plot_cluster(scvi, 'scVI', y_true, y_true_int, "true", axs[6][0])
plot_cluster(scgnn, 'scGNN', y_true, y_true_int, "true", axs[6][1])
plot_cluster(adclust, 'ADClust', y_true, y_true_int, "true", axs[6][2])
plot_cluster(scace, 'scAce', y_true, y_true_int, "true", axs[6][3])
plot_cluster(scdac, 'scDAC', y_true, y_true_int, "true", axs[6][4])
plot_cluster(dmvae, 'DMVAE', y_true, y_true_int, "true", axs[6][5])

|S17 (1350,)
Method: scVI, ARI=0.16, NMI=0.5
Method: scGNN, ARI=0.23, NMI=0.57
Method: ADClust, ARI=0.52, NMI=0.59
Method: scAce, ARI=0.34, NMI=0.61
Method: scDAC, ARI=0.37, NMI=0.65
Method: DMVAE, ARI=0.88, NMI=0.79


## Romanov

In [67]:
umap_all = np.load("/Volumes/SSD/MCW/Research/Aim 1/Results/umap_Romanov.npz", allow_pickle=True)['UMAP']
umap_all = umap_all.item()
data_mat = h5py.File('/Volumes/SSD/MCW/Research/Aim 1/Data/Romanov/data.h5')
obs = data_mat['obs']
ds = obs['cell_type1']
print(ds.dtype, ds.shape)
y_true_bytes = np.array(ds)
y_true = y_true_bytes.astype(str)
classes, y_true_int = np.unique(y_true, return_inverse=True)

data_mat.close()

scvi = np.load('/Users/enid/Downloads/Romanov/scvi.npz', allow_pickle=True)
adclust = np.load('/Users/enid/Downloads/Romanov/adclust.npz')
scace = np.load('/Users/enid/Downloads/Romanov/scace.npz', allow_pickle=True)
dmvae = np.load('/Users/enid/Downloads/Romanov/dmvae.npz')
scgnn = np.load('/Users/enid/Downloads/Romanov/scgnn.npz')
scdac = np.load('/Users/enid/Downloads/Romanov/scdac.npz')

plot_cluster(scvi, 'scVI', y_true, y_true_int, "true", axs[7][0])
plot_cluster(scgnn, 'scGNN', y_true, y_true_int, "true", axs[7][1])
plot_cluster(adclust, 'ADClust', y_true, y_true_int, "true", axs[7][2])
plot_cluster(scace, 'scAce', y_true, y_true_int, "true", axs[7][3])
plot_cluster(scdac, 'scDAC', y_true, y_true_int, "true", axs[7][4])
plot_cluster(dmvae, 'DMVAE', y_true, y_true_int, "true", axs[7][5])

|S12 (2881,)
Method: scVI, ARI=0.3, NMI=0.56
Method: scGNN, ARI=0.26, NMI=0.33
Method: ADClust, ARI=0.33, NMI=0.45
Method: scAce, ARI=0.41, NMI=0.57
Method: scDAC, ARI=0.36, NMI=0.53
Method: DMVAE, ARI=0.71, NMI=0.6


## Young

In [ ]:
umap_all = np.load("/Volumes/SSD/MCW/Research/Aim 1/Results/umap_Young.npz", allow_pickle=True)['UMAP']
umap_all = umap_all.item()
data_mat = h5py.File('/Volumes/SSD/MCW/Research/Aim 1/Data/Young/data.h5')
obs = data_mat['obs']
ds = obs['cell_type1']
print(ds.dtype, ds.shape)
y_true_bytes = np.array(ds)
y_true = y_true_bytes.astype(str)
classes, y_true_int = np.unique(y_true, return_inverse=True)

data_mat.close()

scvi = np.load('/Users/enid/Downloads/Young/scvi.npz', allow_pickle=True)
adclust = np.load('/Users/enid/Downloads/Young/adclust.npz')
scace = np.load('/Users/enid/Downloads/Young/scace.npz', allow_pickle=True)
dmvae = np.load('/Users/enid/Downloads/Young/dmvae.npz')
scgnn = np.load('/Users/enid/Downloads/Young/scgnn.npz')
scdac = np.load('/Users/enid/Downloads/Young/scdac.npz')

plot_cluster(scvi, 'scVI', y_true, y_true_int, "true", axs[9][0])
plot_cluster(scgnn, 'scGNN', y_true, y_true_int, "true", axs[9][1])
plot_cluster(adclust, 'ADClust', y_true, y_true_int, "true", axs[9][2])
plot_cluster(scace, 'scAce', y_true, y_true_int, "true", axs[9][3])
plot_cluster(scdac, 'scDAC', y_true, y_true_int, "true", axs[9][4])
plot_cluster(dmvae, 'DMVAE', y_true, y_true_int, "true", axs[9][5])

|S24 (5685,)
Method: scVI, ARI=0.47, NMI=0.72
Method: scGNN, ARI=0.23, NMI=0.37
Method: ADClust, ARI=0.39, NMI=0.57
Method: scAce, ARI=0.6, NMI=0.74
Method: scDAC, ARI=0.45, NMI=0.63
Method: DMVAE, ARI=0.56, NMI=0.66


## Wang Lung

In [ ]:
umap_all = np.load("/Volumes/SSD/MCW/Research/Aim 1/Results/umap_Wang_Lung.npz", allow_pickle=True)['UMAP']
umap_all = umap_all.item()
data_mat = h5py.File('/Volumes/SSD/MCW/Research/Aim 1/Data/Wang_Lung/data.h5')
obs = data_mat['obs']
ds = obs['cell_type1']
print(ds.dtype, ds.shape)
y_true_bytes = np.array(ds)
y_true = y_true_bytes.astype(str)
classes, y_true_int = np.unique(y_true, return_inverse=True)

data_mat.close()

scvi = np.load('/Users/enid/Downloads/Wang_Lung/scvi.npz', allow_pickle=True)
adclust = np.load('/Users/enid/Downloads/Wang_Lung/adclust.npz')
scace = np.load('/Users/enid/Downloads/Wang_Lung/scace.npz', allow_pickle=True)
dmvae = np.load('/Users/enid/Downloads/Wang_Lung/dmvae.npz')
scgnn = np.load('/Users/enid/Downloads/Wang_Lung/scgnn.npz')
scdac = np.load('/Users/enid/Downloads/Wang_Lung/scdac.npz')

plot_cluster(scvi, 'scVI', y_true, y_true_int, "true", axs[8][0])
plot_cluster(scgnn, 'scGNN', y_true, y_true_int, "true", axs[8][1])
plot_cluster(adclust, 'ADClust', y_true, y_true_int, "true", axs[8][2])
plot_cluster(scace, 'scAce', y_true, y_true_int, "true", axs[8][3])
plot_cluster(scdac, 'scDAC', y_true, y_true_int, "true", axs[8][4])
plot_cluster(dmvae, 'DMVAE', y_true, y_true_int, "true", axs[8][5])

|S16 (9519,)
Method: scVI, ARI=0.14, NMI=0.37
Method: scGNN, ARI=0.22, NMI=0.42
Method: ADClust, ARI=0.37, NMI=0.52
Method: scAce, ARI=0.19, NMI=0.37
Method: scDAC, ARI=0.23, NMI=0.4
Method: DMVAE, ARI=0.96, NMI=0.9


## save the plot

In [84]:
plt.savefig('/Volumes/SSD/MCW/Research/Aim 1/Documents/Paper_draft/papers/umap_true_suppl.png', dpi=600, format='png', bbox_inches='tight')